## Environment Setting

In [2]:
print("Start instcolorization test")

Start instcolorization test


In [2]:
# !git clone https://github.com/ericsujw/InstColorization.git

# #!sh scripts/download_model.sh
# # error legacy code - manual download
# # from google.colab import drive
# # drive.mount('/content/drive')
import gdown

# # File 1
# url1 = "https://drive.google.com/uc?id=1KoJTi_sUliv4unX0oKU4LrmqW848lg0c"
# gdown.download(url1, quiet=False)

# File 2
# url2 = "https://drive.google.com/uc?id=1dXda8H1WcxwHvlLZNicjwiRchMUTUXwA"
# gdown.download(url2, quiet=False)

# print("Download complete!")


# !cp checkpoints.zip ./InstColorization/checkpoints.zip
# !cp ViCoWDataset.zip ./InstColorization/ViCoWDataset.zip

In [4]:
# #!sh scripts/download_model.sh
# # error legacy code - manual download
# import zipfile

# with zipfile.ZipFile('checkpoints.zip', 'r') as zf:
#     zf.extractall()

# with zipfile.ZipFile('ViCoWDataset.zip', 'r') as zf:
#     zf.extractall()

In [5]:
# Updated PyTorch, torchvision, and detectron2 versions for compatibility with current Colab environments.
# Original versions torch==1.5 and detectron2==0.1.2 are too old and cause installation errors.
# PyYAML and dominate version constraints removed to resolve installation issues.
!pip install -U torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install -U gradio # cython pyyaml visdom dominate gdown wheel setuptools cython numpy scikit-image 



  Using cached gradio-6.2.0-py3-none-any.whl.metadata (16 kB)
  Using cached aiofiles-24.1.0-py3-none-any.whl.metadata (10 kB)
  Using cached brotli-1.2.0-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (6.1 kB)
  Using cached ffmpy-1.0.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached gradio_client-2.0.2-py3-none-any.whl.metadata (7.1 kB)
  Using cached groovy-0.1.2-py3-none-any.whl.metadata (6.1 kB)
  Using cached huggingface_hub-1.2.3-py3-none-any.whl.metadata (13 kB)
  Using cached orjson-3.11.5-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (41 kB)
  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached pydub-0.25.1-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached python_multipart-0.0.21-py3-none-any.whl.metadata (1.8 kB)
  Using cached safehttpx-0.1.7-py3-none-any.whl.metadata (4.2 kB)
  Using cached seman

In [3]:
# sudo apt update
# sudo apt install -y build-essential python3-dev python3-pip git cython3
!pip install -U 'git+https://github.com/cocodataset/cocoapi.git#subdirectory=PythonAPI'
# Use python -m pip install for detectron2 as it can be more robust for git installs.
!python -m pip install 'detectron2@git+https://github.com/facebookresearch/detectron2.git'

  Cloning https://github.com/cocodataset/cocoapi.git to /tmp/pip-req-build-9pyuvuh7
  Running command git clone --filter=blob:none --quiet https://github.com/cocodataset/cocoapi.git /tmp/pip-req-build-9pyuvuh7
  Resolved https://github.com/cocodataset/cocoapi.git to commit 8c9bcc3cf640524c4c20a9c40e89cb6a2f2fa0e9
  Preparing metadata (setup.py) ... done
  Created wheel for pycocotools: filename=pycocotools-2.0-cp312-cp312-linux_x86_64.whl size=424531 sha256=fff982e79d093ddce12ee6f171fde4c95640ea348e6a9dae0e9ab0033841245c
  Stored in directory: /tmp/pip-ephem-wheel-cache-u_arm3ul/wheels/95/e6/c7/8ceda667bca7218619fea052622a0b11a37fb51c28c993fae3
Successfully built pycocotools
  Attempting uninstall: pycocotools
    Found existing installation: pycocotools 2.0.11
    Uninstalling pycocotools-2.0.11:
      Successfully uninstalled pycocotools-2.0.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of th

## Start Colorization

In [3]:
cd InstColorization/

/home/h/InstColorization


### Detect Object bounding box


Setting the Detectron2.

In [4]:
# Install detectron2 from source as pre-built wheels are not available for the current PyTorch/CUDA versions.
#!pip install 'git+https://github.com/facebookresearch/detectron2.git'

from os.path import join, isfile, isdir
from os import listdir
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
from argparse import ArgumentParser

import detectron2
from detectron2.utils.logger import setup_logger
setup_logger()

import numpy as np
import cv2

# import some common detectron2 utilities
from detectron2 import model_zoo
from detectron2.engine import DefaultPredictor
from detectron2.config import get_cfg

import torch

cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml"))
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.7
cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x.yaml")
predictor = DefaultPredictor(cfg)

/home/h/312inst/lib/python3.12/site-packages/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


[01/08 09:34:32 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x/139653917/model_final_2d9806.pkl ...


### Colorize Images

In [5]:
!python inference_bbox.py --test_img_dir "test" --filter_no_obj

/home/h/312inst/lib/python3.12/site-packages/detectron2/model_zoo/model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
[01/08 09:34:40 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from https://dl.fbaipublicfiles.com/detectron2/COCO-InstanceSegmentation/mask_rcnn_X_101_32x8d_FPN_3x/139653917/model_final_2d9806.pkl ...
  0%|                                                   | 0/363 [00:00<?, ?it/s]/home/h/312inst/lib/python3.12/site-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4317.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
W0108 09:34:42.483000 1578

In [50]:
!python test_fusion.py --name test_fusion --sample_p 1.0 --model fusion --fineSize 256 --test_img_dir test --results_img_dir test_results

#Testing images = 106
initialize network with normal
initialize network with normal
initialize network with normal
model [FusionModel] was created
load Fusion model from checkpoints/coco_mask/latest_net_GF.pth
100%|█████████████████████████████████████████| 106/106 [01:24<00:00,  1.26it/s]
0 images without bounding boxes


In [54]:
import os
import shutil

# Paths
folder_jpg = "test"
folder_png = "test_results"
folder_done = "test/done"

# Create done folder if not exists
os.makedirs(folder_done, exist_ok=True)

# Get base names (without extension) of png files
png_basenames = {
    os.path.splitext(f)[0]
    for f in os.listdir(folder_png)
    if f.lower().endswith(".png")
}

# Iterate jpg files and move matched ones
for f in os.listdir(folder_jpg):
    if not f.lower().endswith(".jpg"):
        continue

    base_name = os.path.splitext(f)[0]

    if base_name in png_basenames:
        src = os.path.join(folder_jpg, f)
        dst = os.path.join(folder_done, f)
        shutil.move(src, dst)
        print(f"Moved: {f}")

print("Done.")


Moved: VIDEO1_NhungNguoiVietLenHuyenThoai_frame_1696.jpg
Moved: VIDEO2_VaoNamRaBac_frame_0576.jpg
Moved: VIDEO3_MuiCoChay_frame_0027.jpg
Moved: VIDEO2_VaoNamRaBac_frame_0943.jpg
Moved: VIDEO1_NhungNguoiVietLenHuyenThoai_frame_1087.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_0542.jpg
Moved: VIDEO2_VaoNamRaBac_frame_0618.jpg
Moved: VIDEO1_NhungNguoiVietLenHuyenThoai_frame_1680.jpg
Moved: VIDEO2_VaoNamRaBac_frame_1338.jpg
Moved: VIDEO1_NhungNguoiVietLenHuyenThoai_frame_1142.jpg
Moved: VIDEO2_VaoNamRaBac_frame_0776.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_1302.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_0745.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_1170.jpg
Moved: VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0393.jpg
Moved: VIDEO2_VaoNamRaBac_frame_0250.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_0909.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_0755.jpg
Moved: VIDEO3_MuiCoChay_frame_0453.jpg
Moved: VIDEO4_HaNoi12NgayDem_frame_0346.jpg
Moved: VIDEO3_MuiCoChay_frame_1256.jpg
Moved: VIDEO3_MuiCoChay_frame_0991.jpg
Moved

In [52]:
%%capture
!pip install lpips

In [4]:
import os
import cv2
import torch
import numpy as np
import lpips
from skimage import color
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

# ===================== SETUP =====================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
loss_fn_vgg = lpips.LPIPS(net="vgg").to(device)

valid_extensions = [".png", ".jpg", ".jpeg", ".bmp"]

# ===================== METRICS =====================
def calculate_cf(img_gt, img_out):
    """
    Color Fidelity (CF) = PSNR trên kênh a,b (Lab)
    img_gt, img_out: BGR uint8 [0,255]
    return: CF (dB)
    """
    img_gt = cv2.cvtColor(img_gt, cv2.COLOR_BGR2RGB)
    img_out = cv2.cvtColor(img_out, cv2.COLOR_BGR2RGB)

    lab_gt = color.rgb2lab(img_gt)
    lab_out = color.rgb2lab(img_out)

    ab_gt = lab_gt[:, :, 1:3]
    ab_out = lab_out[:, :, 1:3]

    mse = np.mean((ab_gt - ab_out) ** 2)
    if mse == 0:
        return float("inf")

    return 10 * np.log10((255.0 ** 2) / mse)


def im2tensor(image):
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    image = torch.from_numpy(image).permute(2, 0, 1).float()
    image = (image / 127.5) - 1.0
    return image.unsqueeze(0).to(device)

# ===================== EVAL =====================
def calculate_full_metrics(gt_dir, out_dir):
    gt_images = sorted(
        [f for f in os.listdir(gt_dir) if f.lower().endswith((".png", ".jpg", ".jpeg"))]
    )

    m = {"psnr": [], "ssim": [], "cf": [], "lpips": []}

    print(f"--- Đang đánh giá {len(gt_images)} ảnh trên {device} ---")

    for filename in gt_images:
        path_gt = os.path.join(gt_dir, filename)

        # --- tìm file output khác đuôi ---
        basename = os.path.splitext(filename)[0]
        path_out = None
        for ext in valid_extensions:
            temp_path = os.path.join(out_dir, basename + ext)
            if os.path.exists(temp_path):
                path_out = temp_path
                break

        if path_out is None:
            continue

        img_gt = cv2.imread(path_gt)
        img_out = cv2.imread(path_out)

        if img_gt is None or img_out is None:
            continue

        if img_gt.shape != img_out.shape:
            img_out = cv2.resize(img_out, (img_gt.shape[1], img_gt.shape[0]))

        # PSNR & SSIM
        m["psnr"].append(psnr(img_gt, img_out, data_range=255))
        m["ssim"].append(ssim(img_gt, img_out, channel_axis=2, data_range=255))

        # CF
        cf_val = calculate_cf(img_gt, img_out)
        m["cf"].append(cf_val)

        # LPIPS
        with torch.no_grad():
            m["lpips"].append(
                loss_fn_vgg(im2tensor(img_gt), im2tensor(img_out)).item()
            )

        print(
            f"[{filename}] "
            f"PSNR: {m['psnr'][-1]:.2f} | "
            f"SSIM: {m['ssim'][-1]:.4f} | "
            f"CF: {cf_val:.2f} dB | "
            f"LPIPS: {m['lpips'][-1]:.4f}"
        )

    if m["psnr"]:
        print("\n" + "=" * 50)
        print("KẾT QUẢ TRUNG BÌNH TOÀN BỘ TẬP TEST")
        print(f"PSNR : {np.mean(m['psnr']):.2f} dB ↑")
        print(f"SSIM : {np.mean(m['ssim']):.4f} ↑")
        print(f"CF   : {np.mean(m['cf']):.2f} dB ↑")
        print(f"LPIPS: {np.mean(m['lpips']):.4f} ↓")
        print("=" * 50)
    else:
        print("Không có dữ liệu.")

# ===================== MAIN =====================
if __name__ == "__main__":
    folder_gt = "/home/h/DDColor/dataset/test"
    folder_out = "/home/h/InstColorization/test_results"
    calculate_full_metrics(folder_gt, folder_out)

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/home/h/312inst/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/h/312inst/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Loading model from: /home/h/312inst/lib/python3.12/site-packages/lpips/weights/v0.1/vgg.pth
--- Đang đánh giá 382 ảnh trên cuda ---
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0095.jpg] PSNR: 37.13 | SSIM: 0.9739 | CF: 41.08 dB | LPIPS: 0.2231
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0115.jpg] PSNR: 34.90 | SSIM: 0.9567 | CF: 38.16 dB | LPIPS: 0.2997
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0126.jpg] PSNR: 30.11 | SSIM: 0.8654 | CF: 39.67 dB | LPIPS: 0.2552
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0131.jpg] PSNR: 30.02 | SSIM: 0.8614 | CF: 39.09 dB | LPIPS: 0.2640
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0151.jpg] PSNR: 33.67 | SSIM: 0.9424 | CF: 39.54 dB | LPIPS: 0.2567
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0152.jpg] PSNR: 35.82 | SSIM: 0.9608 | CF: 43.73 dB | LPIPS: 0.2695
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0159.jpg] PSNR: 29.85 | SSIM: 0.8392 | CF: 38.45 dB | LPIPS: 0.3222
[VIDEO1_NhungNguoiVietLenHuyenThoai_frame_0178.jpg] PSNR: 33.95 | SSIM: 0.9364 | CF: 40.33 dB | LP

In [14]:
ls

 InstColorization.ipynb
 LICENSE
 README.md
 README_TRAIN.md
'ViCoW A Dataset for Colorization and Restoration of Vietnam War Imagery'/
 ViCoW-test/
 ViCoWDataset.zip
 __pycache__/
 check.py
 checkpoints/
 checkpoints.zip
 data/
 download.py
 env.yml
 example/
 example_bbox/
 fusion_dataset.py
 gradio_demo.py
 image_util.py
 imgs/
 inference_bbox.py
 models/
 options/
 results/
 scripts/
 test/
 test_bbox/
 test_fusion.py
 test_results/
 train.py
 train_data/
 util/


In [1]:
cd InstColorization

/home/h/InstColorization


In [7]:
pwd

'/home/h/InstColorization'

In [4]:
!python gradio_demo.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://588c9bc5b5a66b12fb.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
Keyboard interruption in main thread... closing server.
^C
